# Tutorial: Phase 1 OME-Zarr Open + DatasetSummary

Audience:
- Engineers validating the first Lucida phase-1 deliverable.

Prerequisites:
- Run `uv sync` in this repository.

Learning goals:
- Build a tiny OME-Zarr dataset locally.
- Open it through the service layer and the HTTP endpoint.
- Validate contract expectations with explicit assertions.


## Step 1 - Setup imports and helpers

This cell creates a minimal TCZYX OME-Zarr dataset with two pyramid levels.


In [1]:
from __future__ import annotations

import json
import tempfile
from pathlib import Path

import numpy as np
import zarr
from fastapi.testclient import TestClient

from lucida.server.app import create_app
from lucida.service.dataset_service import DatasetService


def build_tiny_omezarr(dataset_path: Path) -> str:
    dataset_path.mkdir(parents=True, exist_ok=True)
    root = zarr.open_group(store=str(dataset_path), mode='w')

    shape_0 = (1, 2, 4, 8, 10)
    shape_1 = (1, 2, 2, 4, 5)
    root.create_array(
        '0',
        data=np.arange(np.prod(shape_0), dtype=np.uint16).reshape(shape_0),
        chunks=(1, 1, 2, 4, 5),
        overwrite=True,
    )
    root.create_array(
        '1',
        data=np.arange(np.prod(shape_1), dtype=np.uint16).reshape(shape_1),
        chunks=(1, 1, 1, 2, 3),
        overwrite=True,
    )

    root.attrs['multiscales'] = [
        {
            'name': 'primary',
            'axes': [
                {'name': 't', 'type': 't'},
                {'name': 'c', 'type': 'c'},
                {'name': 'z', 'type': 'z', 'unit': 'micron'},
                {'name': 'y', 'type': 'y', 'unit': 'micron'},
                {'name': 'x', 'type': 'x', 'unit': 'micron'},
            ],
            'datasets': [
                {
                    'path': '0',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 1, 1, 1]},
                        {'type': 'translation', 'translation': [0, 0, 0, 0, 0]},
                    ],
                },
                {
                    'path': '1',
                    'coordinateTransformations': [
                        {'type': 'scale', 'scale': [1, 1, 2, 2, 2]}
                    ],
                },
            ],
        }
    ]
    root.attrs['omero'] = {
        'channels': [
            {'index': 0, 'label': 'DNA', 'color': 'FF0000', 'window': {'start': 10, 'end': 400}},
            {'index': 1, 'label': 'RNA', 'color': '00FF00', 'window': {'start': 20, 'end': 200}},
        ]
    }

    return str(dataset_path)


## Step 2 - Create the dataset


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix='lucida-notebook-'))
dataset_uri = build_tiny_omezarr(tmp_dir / 'tiny.zarr')
dataset_uri


'/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-notebook-m8hdkmeo/tiny.zarr'

## Step 3 - Open through the service layer


In [3]:
service = DatasetService()
service_response = service.open_dataset(uri=dataset_uri)
service_payload = service_response.model_dump(mode='json')
print(json.dumps(service_payload['dataset_summary'], indent=2)[:1200])


{
  "schema_version": 1,
  "dataset_id": "ds_46b5ea99e1754864",
  "uri": "file:///private/var/folders/hs/qw7ws1q52153c4c639t_p3600000gn/T/lucida-notebook-m8hdkmeo/tiny.zarr",
  "opened_at": "2026-02-23T22:33:40.574646Z",
  "axes": [
    {
      "name": "t",
      "role": "t",
      "size": 1,
      "unit": null,
      "scale": 1.0,
      "translation": 0.0,
      "direction": 1
    },
    {
      "name": "c",
      "role": "c",
      "size": 2,
      "unit": null,
      "scale": 1.0,
      "translation": 0.0,
      "direction": 1
    },
    {
      "name": "z",
      "role": "z",
      "size": 4,
      "unit": "micron",
      "scale": 1.0,
      "translation": 0.0,
      "direction": 1
    },
    {
      "name": "y",
      "role": "y",
      "size": 8,
      "unit": "micron",
      "scale": 1.0,
      "translation": 0.0,
      "direction": 1
    },
    {
      "name": "x",
      "role": "x",
      "size": 10,
      "unit": "micron",
      "scale": 1.0,
      "translation": 0.0,
      "

In [4]:
summary = service_response.dataset_summary
assert summary.schema_version == 1
assert summary.dataset_id
assert summary.axes
assert all(axis.size > 0 for axis in summary.axes)
assert summary.multiscales and summary.multiscales[0].levels
assert summary.shape == [1, 2, 4, 8, 10]
assert summary.dtype == 'uint16'
print('Service assertions passed.')


Service assertions passed.


## Step 4 - Open through the HTTP endpoint

This uses `fastapi.testclient` so no external server process is needed.


In [5]:
app = create_app()
with TestClient(app) as client:
    http_response = client.post('/dataset/open', json={'schema_version': 1, 'uri': dataset_uri})

assert http_response.status_code == 200
http_payload = http_response.json()
assert http_payload['schema_version'] == 1
assert http_payload['dataset_summary']['schema_version'] == 1
assert http_payload['dataset_summary']['dataset_id']
assert len(http_payload['dataset_summary']['axes']) > 0
assert len(http_payload['dataset_summary']['multiscales']) >= 1
assert len(http_payload['dataset_summary']['multiscales'][0]['levels']) >= 1
print('HTTP assertions passed.')


HTTP assertions passed.


## Expected output checks

After running top-to-bottom, verify:
- The service summary JSON preview prints with `schema_version: 1`.
- `Service assertions passed.` is printed.
- `HTTP assertions passed.` is printed.
- The dataset summary contains non-empty `dataset_id`, populated `axes`, and at least one multiscale level.
